<a href="https://colab.research.google.com/github/Lyna122/Data-analytic/blob/main/time_series_forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prévision de séries temporelles
## Analyse et modélisation des ventes avec détection de tendances et saisonnalités

In [2]:
# Installation des packages nécessaires
import sys
!{sys.executable} -m pip install pandas numpy matplotlib seaborn scikit-learn statsmodels plotly --quiet

In [3]:
# Import des bibliothèques
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')

# Configuration des graphiques
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("Bibliothèques chargées avec succès")

Bibliothèques chargées avec succès


## 1. Chargement des données

Utilisation du dataset Superstore Sales pour l'analyse des ventes

In [13]:
# Chargement des données
df = pd.read_csv('/content/superstor_sales.csv', encoding='latin-1')

print(f"Dimensions du dataset: {df.shape}")
print(f"\nPériode couverte: {df['Order Date'].min()} à {df['Order Date'].max()}")
print(f"\nNombre de transactions: {len(df)}")

df.head()

Dimensions du dataset: (9800, 18)

Période couverte: 01/01/2018 à 31/12/2017

Nombre de transactions: 9800


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680


In [16]:
# Préparation des données temporelles
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)
df = df.sort_values('Order Date')

# Agrégation des ventes mensuelles
monthly_sales = df.groupby(pd.Grouper(key='Order Date', freq='M')).agg({
    'Sales': 'sum',
    'Order ID': 'count'
}).reset_index()

monthly_sales.columns = ['Date', 'Sales', 'Orders']
monthly_sales = monthly_sales[monthly_sales['Sales'] > 0]

print(f"Nombre de mois analysés: {len(monthly_sales)}")
print(f"\nStatistiques des ventes mensuelles:")
print(monthly_sales['Sales'].describe())

Nombre de mois analysés: 48

Statistiques des ventes mensuelles:
count        48.000000
mean      47115.349640
std       24978.687305
min        4519.892000
25%       29621.712625
50%       39202.126500
75%       64391.969125
max      117938.155000
Name: Sales, dtype: float64


## 2. Analyse exploratoire de la série temporelle

In [17]:
# Visualisation de la série temporelle complète
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=monthly_sales['Date'],
    y=monthly_sales['Sales'],
    mode='lines+markers',
    name='Ventes mensuelles',
    line=dict(color='#3b82f6', width=2),
    marker=dict(size=6)
))

fig.update_layout(
    title='Évolution des ventes mensuelles',
    xaxis_title='Date',
    yaxis_title='Ventes ($)',
    height=500,
    hovermode='x unified'
)

fig.show()

# Statistiques descriptives
print("\nTendance générale:")
print(f"Ventes moyennes: ${monthly_sales['Sales'].mean():,.2f}")
print(f"Ventes médiane: ${monthly_sales['Sales'].median():,.2f}")
print(f"Écart-type: ${monthly_sales['Sales'].std():,.2f}")
print(f"Coefficient de variation: {(monthly_sales['Sales'].std() / monthly_sales['Sales'].mean() * 100):.2f}%")


Tendance générale:
Ventes moyennes: $47,115.35
Ventes médiane: $39,202.13
Écart-type: $24,978.69
Coefficient de variation: 53.02%


In [18]:
# Analyse par année
monthly_sales['Year'] = monthly_sales['Date'].dt.year
monthly_sales['Month'] = monthly_sales['Date'].dt.month

yearly_comparison = monthly_sales.groupby('Year')['Sales'].agg(['sum', 'mean', 'count']).reset_index()
yearly_comparison.columns = ['Année', 'Total', 'Moyenne mensuelle', 'Nombre de mois']

print("\nComparaison annuelle:")
print(yearly_comparison.to_string(index=False))

# Visualisation de la comparaison annuelle
fig = go.Figure()

fig.add_trace(go.Bar(
    x=yearly_comparison['Année'],
    y=yearly_comparison['Total'],
    marker_color='#10b981',
    text=yearly_comparison['Total'].apply(lambda x: f'${x/1000:.0f}K'),
    textposition='outside'
))

fig.update_layout(
    title='Ventes totales par année',
    xaxis_title='Année',
    yaxis_title='Ventes totales ($)',
    height=400
)

fig.show()


Comparaison annuelle:
 Année       Total  Moyenne mensuelle  Nombre de mois
  2015 479856.2081       39988.017342              12
  2016 459436.0054       38286.333783              12
  2017 600192.5500       50016.045833              12
  2018 722052.0192       60171.001600              12


## 3. Décomposition de la série temporelle

Séparation des composantes: tendance, saisonnalité et résidus

In [19]:
# Décomposition de la série temporelle
# On utilise une période de 12 mois pour capturer la saisonnalité annuelle
decomposition = seasonal_decompose(
    monthly_sales.set_index('Date')['Sales'],
    model='additive',
    period=12
)

# Extraction des composantes
trend = decomposition.trend
seasonal = decomposition.seasonal
residual = decomposition.resid

# Visualisation des composantes
fig = make_subplots(
    rows=4, cols=1,
    subplot_titles=(
        'Série originale',
        'Tendance',
        'Saisonnalité',
        'Résidus'
    ),
    vertical_spacing=0.08
)

# Série originale
fig.add_trace(
    go.Scatter(x=monthly_sales['Date'], y=monthly_sales['Sales'],
               mode='lines', name='Original', line=dict(color='#3b82f6')),
    row=1, col=1
)

# Tendance
fig.add_trace(
    go.Scatter(x=trend.index, y=trend.values,
               mode='lines', name='Tendance', line=dict(color='#10b981')),
    row=2, col=1
)

# Saisonnalité
fig.add_trace(
    go.Scatter(x=seasonal.index, y=seasonal.values,
               mode='lines', name='Saisonnalité', line=dict(color='#f59e0b')),
    row=3, col=1
)

# Résidus
fig.add_trace(
    go.Scatter(x=residual.index, y=residual.values,
               mode='lines', name='Résidus', line=dict(color='#ef4444')),
    row=4, col=1
)

fig.update_layout(height=1000, showlegend=False)
fig.update_xaxes(title_text="Date", row=4, col=1)
fig.update_yaxes(title_text="Ventes ($)", row=1, col=1)
fig.update_yaxes(title_text="Tendance ($)", row=2, col=1)
fig.update_yaxes(title_text="Effet saisonnier ($)", row=3, col=1)
fig.update_yaxes(title_text="Résidus ($)", row=4, col=1)

fig.show()

In [20]:
# Analyse de la saisonnalité mensuelle
seasonal_pattern = monthly_sales.groupby('Month')['Sales'].mean().reset_index()
seasonal_pattern['Month_Name'] = seasonal_pattern['Month'].apply(
    lambda x: ['Jan', 'Fév', 'Mar', 'Avr', 'Mai', 'Jun',
               'Jul', 'Aoû', 'Sep', 'Oct', 'Nov', 'Déc'][x-1]
)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=seasonal_pattern['Month_Name'],
    y=seasonal_pattern['Sales'],
    marker_color='#8b5cf6',
    text=seasonal_pattern['Sales'].apply(lambda x: f'${x/1000:.0f}K'),
    textposition='outside'
))

fig.update_layout(
    title='Saisonnalité: Ventes moyennes par mois',
    xaxis_title='Mois',
    yaxis_title='Ventes moyennes ($)',
    height=400
)

fig.show()

print("\nMois avec les ventes les plus élevées:")
top_months = seasonal_pattern.nlargest(3, 'Sales')[['Month_Name', 'Sales']]
for _, row in top_months.iterrows():
    print(f"{row['Month_Name']}: ${row['Sales']:,.2f}")

print("\nMois avec les ventes les plus faibles:")
low_months = seasonal_pattern.nsmallest(3, 'Sales')[['Month_Name', 'Sales']]
for _, row in low_months.iterrows():
    print(f"{row['Month_Name']}: ${row['Sales']:,.2f}")


Mois avec les ventes les plus élevées:
Nov: $87,540.43
Déc: $80,370.04
Sep: $75,025.85

Mois avec les ventes les plus faibles:
Fév: $14,842.78
Jan: $23,572.91
Avr: $34,070.75


## 4. Test de stationnarité

Vérification de la stationnarité de la série avec le test de Dickey-Fuller augmenté

In [21]:
# Test de Dickey-Fuller augmenté
def adf_test(series, name=''):
    result = adfuller(series.dropna())
    print(f'Test ADF pour {name}:')
    print(f'  Statistique ADF: {result[0]:.4f}')
    print(f'  p-value: {result[1]:.4f}')
    print(f'  Valeurs critiques:')
    for key, value in result[4].items():
        print(f'    {key}: {value:.3f}')

    if result[1] <= 0.05:
        print(f'  Résultat: La série est stationnaire (p-value < 0.05)')
    else:
        print(f'  Résultat: La série est non-stationnaire (p-value > 0.05)')
    print()

# Test sur la série originale
adf_test(monthly_sales['Sales'], 'série originale')

# Test sur la série différenciée
monthly_sales['Sales_Diff'] = monthly_sales['Sales'].diff()
adf_test(monthly_sales['Sales_Diff'], 'série différenciée')

Test ADF pour série originale:
  Statistique ADF: -4.4161
  p-value: 0.0003
  Valeurs critiques:
    1%: -3.578
    5%: -2.925
    10%: -2.601
  Résultat: La série est stationnaire (p-value < 0.05)

Test ADF pour série différenciée:
  Statistique ADF: -8.7271
  p-value: 0.0000
  Valeurs critiques:
    1%: -3.627
    5%: -2.946
    10%: -2.612
  Résultat: La série est stationnaire (p-value < 0.05)



## 5. Préparation des données pour la modélisation

In [22]:
# Création des features temporelles
monthly_sales['Time_Index'] = range(len(monthly_sales))
monthly_sales['Month_Sin'] = np.sin(2 * np.pi * monthly_sales['Month'] / 12)
monthly_sales['Month_Cos'] = np.cos(2 * np.pi * monthly_sales['Month'] / 12)
monthly_sales['Quarter'] = monthly_sales['Date'].dt.quarter

# Moyennes mobiles
monthly_sales['MA_3'] = monthly_sales['Sales'].rolling(window=3, min_periods=1).mean()
monthly_sales['MA_6'] = monthly_sales['Sales'].rolling(window=6, min_periods=1).mean()

# Lag features
monthly_sales['Sales_Lag1'] = monthly_sales['Sales'].shift(1)
monthly_sales['Sales_Lag3'] = monthly_sales['Sales'].shift(3)
monthly_sales['Sales_Lag12'] = monthly_sales['Sales'].shift(12)

# Suppression des valeurs manquantes créées par les lags
df_model = monthly_sales.dropna().copy()

print(f"Nombre d'observations pour la modélisation: {len(df_model)}")
print(f"\nFeatures créées: {df_model.columns.tolist()}")

Nombre d'observations pour la modélisation: 36

Features créées: ['Date', 'Sales', 'Orders', 'Year', 'Month', 'Sales_Diff', 'Time_Index', 'Month_Sin', 'Month_Cos', 'Quarter', 'MA_3', 'MA_6', 'Sales_Lag1', 'Sales_Lag3', 'Sales_Lag12']


In [23]:
# Division des données: 80% entraînement, 20% test
train_size = int(len(df_model) * 0.8)
train_data = df_model.iloc[:train_size]
test_data = df_model.iloc[train_size:]

print(f"Taille ensemble d'entraînement: {len(train_data)} observations")
print(f"Taille ensemble de test: {len(test_data)} observations")
print(f"\nPériode d'entraînement: {train_data['Date'].min()} à {train_data['Date'].max()}")
print(f"Période de test: {test_data['Date'].min()} à {test_data['Date'].max()}")

Taille ensemble d'entraînement: 28 observations
Taille ensemble de test: 8 observations

Période d'entraînement: 2016-01-31 00:00:00 à 2018-04-30 00:00:00
Période de test: 2018-05-31 00:00:00 à 2018-12-31 00:00:00


## 6. Modélisation: Régression linéaire avec features temporelles

In [24]:
# Sélection des features pour la régression
feature_columns = ['Time_Index', 'Month_Sin', 'Month_Cos', 'Quarter',
                   'MA_3', 'MA_6', 'Sales_Lag1', 'Sales_Lag3']

X_train = train_data[feature_columns]
y_train = train_data['Sales']
X_test = test_data[feature_columns]
y_test = test_data['Sales']

# Entraînement du modèle de régression linéaire
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Prédictions
train_pred_lr = lr_model.predict(X_train)
test_pred_lr = lr_model.predict(X_test)

# Calcul des métriques de performance
train_mse_lr = mean_squared_error(y_train, train_pred_lr)
train_mae_lr = mean_absolute_error(y_train, train_pred_lr)
train_r2_lr = r2_score(y_train, train_pred_lr)

test_mse_lr = mean_squared_error(y_test, test_pred_lr)
test_mae_lr = mean_absolute_error(y_test, test_pred_lr)
test_r2_lr = r2_score(y_test, test_pred_lr)

print("Performance du modèle de régression linéaire:")
print("\nEnsemble d'entraînement:")
print(f"  RMSE: ${np.sqrt(train_mse_lr):,.2f}")
print(f"  MAE: ${train_mae_lr:,.2f}")
print(f"  R² Score: {train_r2_lr:.4f}")

print("\nEnsemble de test:")
print(f"  RMSE: ${np.sqrt(test_mse_lr):,.2f}")
print(f"  MAE: ${test_mae_lr:,.2f}")
print(f"  R² Score: {test_r2_lr:.4f}")

# Erreur en pourcentage
mape_test_lr = np.mean(np.abs((y_test - test_pred_lr) / y_test)) * 100
print(f"  MAPE: {mape_test_lr:.2f}%")

Performance du modèle de régression linéaire:

Ensemble d'entraînement:
  RMSE: $11,177.19
  MAE: $8,841.99
  R² Score: 0.7199

Ensemble de test:
  RMSE: $11,337.73
  MAE: $9,326.95
  R² Score: 0.7778
  MAPE: 14.31%


In [25]:
# Importance des features
feature_importance_lr = pd.DataFrame({
    'Feature': feature_columns,
    'Coefficient': lr_model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

fig = go.Figure(go.Bar(
    x=feature_importance_lr['Coefficient'],
    y=feature_importance_lr['Feature'],
    orientation='h',
    marker_color='#3b82f6'
))

fig.update_layout(
    title='Importance des features (Régression Linéaire)',
    xaxis_title='Coefficient',
    yaxis_title='Feature',
    height=400
)

fig.show()

print("\nCoefficients du modèle:")
print(feature_importance_lr.to_string(index=False))


Coefficients du modèle:
   Feature  Coefficient
   Quarter 15184.753270
 Month_Sin  8834.337925
 Month_Cos -4274.325780
Time_Index   202.865652
      MA_3     1.052568
Sales_Lag1    -0.505567
      MA_6     0.321802
Sales_Lag3     0.004058


## 7. Modélisation: Random Forest Regressor

In [26]:
# Entraînement du modèle Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10)
rf_model.fit(X_train, y_train)

# Prédictions
train_pred_rf = rf_model.predict(X_train)
test_pred_rf = rf_model.predict(X_test)

# Calcul des métriques
train_mse_rf = mean_squared_error(y_train, train_pred_rf)
train_mae_rf = mean_absolute_error(y_train, train_pred_rf)
train_r2_rf = r2_score(y_train, train_pred_rf)

test_mse_rf = mean_squared_error(y_test, test_pred_rf)
test_mae_rf = mean_absolute_error(y_test, test_pred_rf)
test_r2_rf = r2_score(y_test, test_pred_rf)

print("Performance du modèle Random Forest:")
print("\nEnsemble d'entraînement:")
print(f"  RMSE: ${np.sqrt(train_mse_rf):,.2f}")
print(f"  MAE: ${train_mae_rf:,.2f}")
print(f"  R² Score: {train_r2_rf:.4f}")

print("\nEnsemble de test:")
print(f"  RMSE: ${np.sqrt(test_mse_rf):,.2f}")
print(f"  MAE: ${test_mae_rf:,.2f}")
print(f"  R² Score: {test_r2_rf:.4f}")

# Erreur en pourcentage
mape_test_rf = np.mean(np.abs((y_test - test_pred_rf) / y_test)) * 100
print(f"  MAPE: {mape_test_rf:.2f}%")

Performance du modèle Random Forest:

Ensemble d'entraînement:
  RMSE: $6,667.76
  MAE: $5,481.20
  R² Score: 0.9003

Ensemble de test:
  RMSE: $17,132.56
  MAE: $11,424.08
  R² Score: 0.4926
  MAPE: 14.05%


In [27]:
# Importance des features (Random Forest)
feature_importance_rf = pd.DataFrame({
    'Feature': feature_columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

fig = go.Figure(go.Bar(
    x=feature_importance_rf['Importance'],
    y=feature_importance_rf['Feature'],
    orientation='h',
    marker_color='#10b981'
))

fig.update_layout(
    title='Importance des features (Random Forest)',
    xaxis_title='Importance',
    yaxis_title='Feature',
    height=400
)

fig.show()

print("\nImportance des features:")
print(feature_importance_rf.to_string(index=False))


Importance des features:
   Feature  Importance
      MA_3    0.296450
   Quarter    0.214787
Time_Index    0.135356
 Month_Sin    0.108357
 Month_Cos    0.085798
Sales_Lag1    0.064540
      MA_6    0.047410
Sales_Lag3    0.047304


## 8. Comparaison des valeurs réelles et prédites

In [28]:
# Visualisation des prédictions vs valeurs réelles
fig = go.Figure()

# Valeurs réelles
fig.add_trace(go.Scatter(
    x=train_data['Date'],
    y=y_train,
    mode='lines+markers',
    name='Valeurs réelles (train)',
    line=dict(color='#3b82f6', width=2),
    marker=dict(size=6)
))

fig.add_trace(go.Scatter(
    x=test_data['Date'],
    y=y_test,
    mode='lines+markers',
    name='Valeurs réelles (test)',
    line=dict(color='#3b82f6', width=2, dash='dot'),
    marker=dict(size=6)
))

# Prédictions régression linéaire
fig.add_trace(go.Scatter(
    x=train_data['Date'],
    y=train_pred_lr,
    mode='lines',
    name='Prédictions LR (train)',
    line=dict(color='#ef4444', width=2, dash='dash')
))

fig.add_trace(go.Scatter(
    x=test_data['Date'],
    y=test_pred_lr,
    mode='lines',
    name='Prédictions LR (test)',
    line=dict(color='#ef4444', width=3)
))

# Prédictions Random Forest
fig.add_trace(go.Scatter(
    x=test_data['Date'],
    y=test_pred_rf,
    mode='lines',
    name='Prédictions RF (test)',
    line=dict(color='#10b981', width=3)
))

fig.update_layout(
    title='Comparaison: Valeurs réelles vs Prédictions',
    xaxis_title='Date',
    yaxis_title='Ventes ($)',
    height=600,
    hovermode='x unified'
)

fig.show()

In [29]:
# Graphique des résidus
residuals_lr = y_test - test_pred_lr
residuals_rf = y_test - test_pred_rf

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Résidus: Régression Linéaire', 'Résidus: Random Forest')
)

# Résidus LR
fig.add_trace(
    go.Scatter(
        x=test_data['Date'],
        y=residuals_lr,
        mode='markers',
        marker=dict(color='#ef4444', size=8),
        name='Résidus LR'
    ),
    row=1, col=1
)

fig.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=1)

# Résidus RF
fig.add_trace(
    go.Scatter(
        x=test_data['Date'],
        y=residuals_rf,
        mode='markers',
        marker=dict(color='#10b981', size=8),
        name='Résidus RF'
    ),
    row=1, col=2
)

fig.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=2)

fig.update_xaxes(title_text="Date", row=1, col=1)
fig.update_xaxes(title_text="Date", row=1, col=2)
fig.update_yaxes(title_text="Résidus ($)", row=1, col=1)
fig.update_yaxes(title_text="Résidus ($)", row=1, col=2)

fig.update_layout(height=400, showlegend=False)
fig.show()

print("Analyse des résidus:")
print("\nRégression Linéaire:")
print(f"  Moyenne des résidus: ${residuals_lr.mean():,.2f}")
print(f"  Écart-type des résidus: ${residuals_lr.std():,.2f}")

print("\nRandom Forest:")
print(f"  Moyenne des résidus: ${residuals_rf.mean():,.2f}")
print(f"  Écart-type des résidus: ${residuals_rf.std():,.2f}")

Analyse des résidus:

Régression Linéaire:
  Moyenne des résidus: $1,118.47
  Écart-type des résidus: $12,061.42

Random Forest:
  Moyenne des résidus: $10,968.15
  Écart-type des résidus: $14,070.20


## 9. Comparaison des modèles

In [30]:
# Tableau comparatif des performances
comparison = pd.DataFrame({
    'Modèle': ['Régression Linéaire', 'Random Forest'],
    'RMSE Train': [np.sqrt(train_mse_lr), np.sqrt(train_mse_rf)],
    'RMSE Test': [np.sqrt(test_mse_lr), np.sqrt(test_mse_rf)],
    'MAE Test': [test_mae_lr, test_mae_rf],
    'R² Test': [test_r2_lr, test_r2_rf],
    'MAPE Test': [mape_test_lr, mape_test_rf]
})

print("Comparaison des performances des modèles:")
print(comparison.to_string(index=False))

# Visualisation comparative
metrics = ['RMSE Test', 'MAE Test', 'R² Test', 'MAPE Test']
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=metrics
)

positions = [(1,1), (1,2), (2,1), (2,2)]
colors = ['#ef4444', '#10b981']

for idx, (metric, pos) in enumerate(zip(metrics, positions)):
    fig.add_trace(
        go.Bar(
            x=comparison['Modèle'],
            y=comparison[metric],
            marker_color=colors,
            showlegend=False
        ),
        row=pos[0], col=pos[1]
    )

fig.update_layout(height=600, title_text="Comparaison des métriques de performance")
fig.show()

Comparaison des performances des modèles:
             Modèle   RMSE Train    RMSE Test     MAE Test  R² Test  MAPE Test
Régression Linéaire 11177.186637 11337.730507  9326.954735 0.777787  14.312468
      Random Forest  6667.756908 17132.557256 11424.075295 0.492588  14.053789


## 10. Prévisions futures

In [31]:
# Création de données pour les 6 prochains mois
last_date = df_model['Date'].max()
future_dates = pd.date_range(start=last_date + pd.DateOffset(months=1), periods=6, freq='M')

future_df = pd.DataFrame({
    'Date': future_dates,
    'Time_Index': range(len(df_model), len(df_model) + 6),
    'Month': future_dates.month,
    'Quarter': future_dates.quarter,
    'Year': future_dates.year
})

future_df['Month_Sin'] = np.sin(2 * np.pi * future_df['Month'] / 12)
future_df['Month_Cos'] = np.cos(2 * np.pi * future_df['Month'] / 12)

# Utiliser les dernières valeurs connues pour les lags et moyennes mobiles
last_sales = df_model['Sales'].iloc[-12:].values
future_df['Sales_Lag1'] = np.concatenate([[last_sales[-1]], [np.nan] * 5])
future_df['Sales_Lag3'] = np.concatenate([[last_sales[-3]], [np.nan] * 5])
future_df['MA_3'] = df_model['MA_3'].iloc[-1]
future_df['MA_6'] = df_model['MA_6'].iloc[-1]

# Pour simplifier, on remplit les NaN avec les dernières valeurs
future_df = future_df.fillna(method='ffill')

# Prédictions
X_future = future_df[feature_columns]
future_pred_lr = lr_model.predict(X_future)
future_pred_rf = rf_model.predict(X_future)

# Visualisation des prévisions
fig = go.Figure()

# Données historiques
fig.add_trace(go.Scatter(
    x=df_model['Date'],
    y=df_model['Sales'],
    mode='lines+markers',
    name='Données historiques',
    line=dict(color='#3b82f6', width=2)
))

# Prévisions LR
fig.add_trace(go.Scatter(
    x=future_df['Date'],
    y=future_pred_lr,
    mode='lines+markers',
    name='Prévisions LR',
    line=dict(color='#ef4444', width=2, dash='dash'),
    marker=dict(size=8)
))

# Prévisions RF
fig.add_trace(go.Scatter(
    x=future_df['Date'],
    y=future_pred_rf,
    mode='lines+markers',
    name='Prévisions RF',
    line=dict(color='#10b981', width=2, dash='dash'),
    marker=dict(size=8)
))

fig.update_layout(
    title='Prévisions des ventes pour les 6 prochains mois',
    xaxis_title='Date',
    yaxis_title='Ventes ($)',
    height=600,
    hovermode='x unified'
)

fig.show()

print("Prévisions pour les 6 prochains mois:")
forecast_comparison = pd.DataFrame({
    'Date': future_df['Date'].dt.strftime('%Y-%m'),
    'Prévision LR': future_pred_lr,
    'Prévision RF': future_pred_rf
})
print(forecast_comparison.to_string(index=False))

Prévisions pour les 6 prochains mois:
   Date  Prévision LR  Prévision RF
2019-01  67753.327945  50390.034015
2019-02  72754.297522  48962.269277
2019-03  76277.902920  51517.039848
2019-04  92619.107875  49976.433747
2019-05  91152.893239  50023.983187
2019-06  87511.240999  50879.027025


## 11. Synthèse et conclusions

In [32]:
print("SYNTHÈSE DE L'ANALYSE")
print()
print("1. CARACTÉRISTIQUES DE LA SÉRIE TEMPORELLE")
print(f"   - Période analysée: {monthly_sales['Date'].min().strftime('%Y-%m')} à {monthly_sales['Date'].max().strftime('%Y-%m')}")
print(f"   - Nombre d'observations: {len(monthly_sales)}")
print(f"   - Ventes moyennes mensuelles: ${monthly_sales['Sales'].mean():,.2f}")
print(f"   - Coefficient de variation: {(monthly_sales['Sales'].std() / monthly_sales['Sales'].mean() * 100):.2f}%")

print("\n2. DÉTECTION DE TENDANCE ET SAISONNALITÉ")
print(f"   - Tendance: Croissante sur la période analysée")
print(f"   - Saisonnalité: Cycle annuel détecté (période = 12 mois)")
print(f"   - Mois à forte activité: {', '.join(seasonal_pattern.nlargest(3, 'Sales')['Month_Name'].tolist())}")
print(f"   - Mois à faible activité: {', '.join(seasonal_pattern.nsmallest(3, 'Sales')['Month_Name'].tolist())}")

print("\n3. PERFORMANCE DES MODÈLES")
print("   Régression Linéaire:")
print(f"     - R² Test: {test_r2_lr:.4f}")
print(f"     - MAPE Test: {mape_test_lr:.2f}%")
print("   Random Forest:")
print(f"     - R² Test: {test_r2_rf:.4f}")
print(f"     - MAPE Test: {mape_test_rf:.2f}%")

if test_r2_rf > test_r2_lr:
    print(f"\n   Meilleur modèle: Random Forest (R² supérieur de {(test_r2_rf - test_r2_lr):.4f})")
else:
    print(f"\n   Meilleur modèle: Régression Linéaire (R² supérieur de {(test_r2_lr - test_r2_rf):.4f})")

print("\n4. VARIABLES PRÉDICTIVES PRINCIPALES")
top_features = feature_importance_rf.head(3)
for idx, row in top_features.iterrows():
    print(f"   - {row['Feature']}: {row['Importance']:.4f}")

print("\n5. PRÉVISIONS")
print(f"   - Ventes prévues (6 prochains mois):")
print(f"     Régression Linéaire: ${future_pred_lr.sum():,.2f}")
print(f"     Random Forest: ${future_pred_rf.sum():,.2f}")

print("\nAnalyse terminée avec succès")

SYNTHÈSE DE L'ANALYSE

1. CARACTÉRISTIQUES DE LA SÉRIE TEMPORELLE
   - Période analysée: 2015-01 à 2018-12
   - Nombre d'observations: 48
   - Ventes moyennes mensuelles: $47,115.35
   - Coefficient de variation: 53.02%

2. DÉTECTION DE TENDANCE ET SAISONNALITÉ
   - Tendance: Croissante sur la période analysée
   - Saisonnalité: Cycle annuel détecté (période = 12 mois)
   - Mois à forte activité: Nov, Déc, Sep
   - Mois à faible activité: Fév, Jan, Avr

3. PERFORMANCE DES MODÈLES
   Régression Linéaire:
     - R² Test: 0.7778
     - MAPE Test: 14.31%
   Random Forest:
     - R² Test: 0.4926
     - MAPE Test: 14.05%

   Meilleur modèle: Régression Linéaire (R² supérieur de 0.2852)

4. VARIABLES PRÉDICTIVES PRINCIPALES
   - MA_3: 0.2965
   - Quarter: 0.2148
   - Time_Index: 0.1354

5. PRÉVISIONS
   - Ventes prévues (6 prochains mois):
     Régression Linéaire: $488,068.77
     Random Forest: $301,748.79

Analyse terminée avec succès
